### Import Dependencies

In [1]:
import openai
import pandas as pd
import cohere

from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import VectorParams, Distance, SparseVectorParams, Modifier, PayloadSchemaType, PointStruct, Document, Prefetch, FusionQuery

In [2]:
from dotenv import load_dotenv

load_dotenv("../../.env")

True

### Retrieval

In [5]:
query = "What is a good summer song?"

In [6]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [7]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

In [8]:
def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-hybrid-search",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )
        ],
        query=models.RrfQuery(rrf=models.Rrf(weights=[3,1])),
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [11]:
results = retrieve_data(query, k=20)

In [12]:
results

{'retrieved_context_ids': ['B09XGWCB4B',
  'B09XGWCB4B',
  'B09Y4X2XTS',
  'B09Y4X2XTS',
  'B0BV5WY5TC',
  'B09NFQ6593',
  'B09NFQ6593',
  'B0BV5WY5TC',
  'B0BXPRDBVN',
  'B0C1SWKV9J',
  'B0BXPRDBVN',
  'B0C1SWKV9J',
  'B0BC9V4LPV',
  'B0B59TJ7RK',
  'B0B59TJ7RK',
  'B0BC9V4LPV',
  'B0BLZG2YMG',
  'B0BLZG2YMG',
  'B0B14D2HZ6',
  'B09RG97953'],
 'retrieved_context': ['Sounds Of Summer: The Very Best Of The Beach Boys[Remastered] ',
  'Sounds Of Summer: The Very Best Of The Beach Boys[Remastered] ',
  'Songs About You ',
  'Songs About You ',
  'That! Feels Good![LP] ',
  'Summer Of Soul ...Or, When The Revolution Could Not Be Televised Soundtrack ',
  'Summer Of Soul ...Or, When The Revolution Could Not Be Televised Soundtrack ',
  'That! Feels Good![LP] ',
  'Life Is Like A Song[LP] ',
  'Stick Season       Explicit Lyrics ',
  'Life Is Like A Song[LP] ',
  'Stick Season       Explicit Lyrics ',
  'good kid, m.A.A.d city (10th Anniversary Edition) [2 LP]       Explicit Lyrics ',
  'The

### Reranking

In [13]:
cohere_client = cohere.ClientV2()

In [14]:
to_rerank = results["retrieved_context"]

In [15]:
to_rerank

['Sounds Of Summer: The Very Best Of The Beach Boys[Remastered] ',
 'Sounds Of Summer: The Very Best Of The Beach Boys[Remastered] ',
 'Songs About You ',
 'Songs About You ',
 'That! Feels Good![LP] ',
 'Summer Of Soul ...Or, When The Revolution Could Not Be Televised Soundtrack ',
 'Summer Of Soul ...Or, When The Revolution Could Not Be Televised Soundtrack ',
 'That! Feels Good![LP] ',
 'Life Is Like A Song[LP] ',
 'Stick Season       Explicit Lyrics ',
 'Life Is Like A Song[LP] ',
 'Stick Season       Explicit Lyrics ',
 'good kid, m.A.A.d city (10th Anniversary Edition) [2 LP]       Explicit Lyrics ',
 'The Best Of Tommy James & The Shondells Crystal Blue Persuasion Anniversary ',
 'The Best Of Tommy James & The Shondells Crystal Blue Persuasion Anniversary ',
 'good kid, m.A.A.d city (10th Anniversary Edition) [2 LP]       Explicit Lyrics ',
 'Listen To The Music ',
 'Listen To The Music ',
 'Donna Summer: 40th Anniversary Picture ',
 'Dawn FM by The Weekend Autograph       expli

In [16]:
query

'What is a good summer song?'

In [17]:
response = cohere_client.rerank(
    model="rerank-v4.0-pro",
    query=query,
    documents=to_rerank,
    top_n=20
)

In [18]:
response

V2RerankResponse(id='e0fb847f-3755-438d-9b35-77a63653d895', results=[V2RerankResponseResultsItem(index=0, relevance_score=0.63847464), V2RerankResponseResultsItem(index=1, relevance_score=0.63847464), V2RerankResponseResultsItem(index=16, relevance_score=0.52887404), V2RerankResponseResultsItem(index=17, relevance_score=0.52887404), V2RerankResponseResultsItem(index=19, relevance_score=0.52887404), V2RerankResponseResultsItem(index=5, relevance_score=0.5210812), V2RerankResponseResultsItem(index=6, relevance_score=0.5210812), V2RerankResponseResultsItem(index=13, relevance_score=0.50546855), V2RerankResponseResultsItem(index=14, relevance_score=0.50546855), V2RerankResponseResultsItem(index=18, relevance_score=0.48203897), V2RerankResponseResultsItem(index=2, relevance_score=0.47813892), V2RerankResponseResultsItem(index=3, relevance_score=0.47813892), V2RerankResponseResultsItem(index=4, relevance_score=0.42403027), V2RerankResponseResultsItem(index=7, relevance_score=0.42403027), V2R

In [19]:
reranked_results = [to_rerank[result.index] for result in response.results]

In [20]:
reranked_results

['Sounds Of Summer: The Very Best Of The Beach Boys[Remastered] ',
 'Sounds Of Summer: The Very Best Of The Beach Boys[Remastered] ',
 'Listen To The Music ',
 'Listen To The Music ',
 'Dawn FM by The Weekend Autograph       explicit_lyrics ',
 'Summer Of Soul ...Or, When The Revolution Could Not Be Televised Soundtrack ',
 'Summer Of Soul ...Or, When The Revolution Could Not Be Televised Soundtrack ',
 'The Best Of Tommy James & The Shondells Crystal Blue Persuasion Anniversary ',
 'The Best Of Tommy James & The Shondells Crystal Blue Persuasion Anniversary ',
 'Donna Summer: 40th Anniversary Picture ',
 'Songs About You ',
 'Songs About You ',
 'That! Feels Good![LP] ',
 'That! Feels Good![LP] ',
 'good kid, m.A.A.d city (10th Anniversary Edition) [2 LP]       Explicit Lyrics ',
 'good kid, m.A.A.d city (10th Anniversary Edition) [2 LP]       Explicit Lyrics ',
 'Life Is Like A Song[LP] ',
 'Life Is Like A Song[LP] ',
 'Stick Season       Explicit Lyrics ',
 'Stick Season       Expli